In [1]:
import pandas as pd
import sqlite3

# Load your cleaned dataset
df = pd.read_csv('../data/ecomm_data_cleaned.csv')

# Create a SQLite database file
conn = sqlite3.connect('../data/ecommerce.db')

# Write the dataframe into a SQL table
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Data imported successfully!")
print("Rows:", len(df))

FileNotFoundError: [Errno 2] No such file or directory: '../data/ecomm_data_cleaned.csv'

In [2]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\HP\OneDrive\Desktop\apexplanet-data-analytics
['.git', '.ipynb_checkpoints', '01_eda.ipynb', '02_sql_extraction.ipynb', 'dashboards', 'data', 'notebooks', 'README.md', 'reports', 'scripts']


In [3]:
import pandas as pd
import sqlite3

# Load your cleaned dataset
df = pd.read_csv('data/ecomm_data_cleaned.csv')

# Create a SQLite database file
conn = sqlite3.connect('data/ecommerce.db')

# Write the dataframe into a SQL table
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Data imported successfully!")
print("Rows:", len(df))

Data imported successfully!
Rows: 51290


In [4]:
query = "SELECT * FROM orders LIMIT 5;"
result = pd.read_sql(query, conn)
result

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


In [5]:
query = """
SELECT "Order ID", "Customer Name", Category, Sales
FROM orders
WHERE Category = 'Technology'
LIMIT 10;
"""
pd.read_sql(query, conn)

,Order ID,Customer Name,Category,Sales
0,CA-2012-124891,Rick Hansen,Technology,2309.650
1,IN-2013-71249,Craig Reiter,Technology,5175.171
2,ES-2013-1579342,Katherine Murray,Technology,2892.510
3,SG-2013-4320,Rick Hansen,Technology,2832.960
4,IN-2013-42360,Jim Mitchum,Technology,2862.675
5,SA-2011-1830,Magdelene Morse,Technology,2616.960
6,CA-2014-143567,Thomas Boland,Technology,2249.910
7,IN-2014-11763,Jim Sink,Technology,2565.594
8,CA-2011-154627,Sue Ann Reed,Technology,2735.952
9,US-2014-133193,Naresj Patel,Technology,1713.840


In [6]:
query = """
SELECT "Order ID", "Customer Name", Sales
FROM orders
ORDER BY Sales DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,Order ID,Customer Name,Sales
0,CA-2011-145317,Sean Miller,22638.480
1,CA-2013-118689,Tamara Chand,17499.950
2,CA-2014-140151,Raymond Buch,13999.960
3,CA-2014-127180,Tom Ashbrook,11199.968
4,CA-2014-166709,Hunter Lopez,10499.970
5,CA-2013-117121,Adrian Barton,9892.740
6,CA-2011-116904,Sanjit Chand,9449.950
7,US-2013-107440,Bill Shonely,9099.930
8,CA-2013-158841,Sanjit Engle,8749.950
9,CA-2013-143714,Christopher Conant,8399.976


In [7]:
query = """
SELECT Category, SUM(Sales) AS total_sales, COUNT(*) AS order_count
FROM orders
GROUP BY Category
ORDER BY total_sales DESC;
"""
pd.read_sql(query, conn)

,Category,total_sales,order_count
0,Technology,4.744557e+06,10141
1,Furniture,4.110874e+06,9876
2,Office Supplies,3.787070e+06,31273


In [8]:
query = """
SELECT "Sub-Category", AVG(Profit) AS avg_profit
FROM orders
GROUP BY "Sub-Category"
HAVING AVG(Profit) < 0;
"""
pd.read_sql(query, conn)

,Sub-Category,avg_profit
0,Tables,-74.429023


In [9]:
# Create a simple region-to-manager lookup table for JOIN practice
import pandas as pd

region_managers = pd.DataFrame({
    'Region': df['Region'].unique(),
    'Manager': [f'Manager_{i+1}' for i in range(df['Region'].nunique())]
})
region_managers.to_sql('region_managers', conn, if_exists='replace', index=False)
region_managers

,Region,Manager
0,East,Manager_1
1,Oceania,Manager_2
2,Central,Manager_3
3,Africa,Manager_4
4,West,Manager_5
5,South,Manager_6
6,Central Asia,Manager_7
7,EMEA,Manager_8
8,North Asia,Manager_9
9,North,Manager_10


In [10]:
query = """
SELECT o.Region, o."Order ID", o.Sales, r.Manager
FROM orders o
JOIN region_managers r ON o.Region = r.Region
LIMIT 10;
"""
pd.read_sql(query, conn)

,Region,Order ID,Sales,Manager
0,East,CA-2012-124891,2309.650,Manager_1
1,Oceania,IN-2013-77878,3709.395,Manager_2
2,Oceania,IN-2013-71249,5175.171,Manager_2
3,Central,ES-2013-1579342,2892.510,Manager_3
4,Africa,SG-2013-4320,2832.960,Manager_4
5,Oceania,IN-2013-42360,2862.675,Manager_2
6,Oceania,IN-2011-81826,1822.080,Manager_2
7,Oceania,IN-2012-86369,5244.840,Manager_2
8,West,CA-2014-135909,5083.960,Manager_5
9,South,CA-2012-116638,4297.644,Manager_6


In [11]:
query = """
SELECT "Order ID", "Customer Name", Sales
FROM orders
WHERE Sales > (SELECT AVG(Sales) FROM orders)
ORDER BY Sales DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,Order ID,Customer Name,Sales
0,CA-2011-145317,Sean Miller,22638.480
1,CA-2013-118689,Tamara Chand,17499.950
2,CA-2014-140151,Raymond Buch,13999.960
3,CA-2014-127180,Tom Ashbrook,11199.968
4,CA-2014-166709,Hunter Lopez,10499.970
5,CA-2013-117121,Adrian Barton,9892.740
6,CA-2011-116904,Sanjit Chand,9449.950
7,US-2013-107440,Bill Shonely,9099.930
8,CA-2013-158841,Sanjit Engle,8749.950
9,CA-2013-143714,Christopher Conant,8399.976


In [12]:
query = """
WITH category_totals AS (
    SELECT Category, SUM(Sales) AS total_sales
    FROM orders
    GROUP BY Category
)
SELECT * FROM category_totals
ORDER BY total_sales DESC;
"""
pd.read_sql(query, conn)

,Category,total_sales
0,Technology,4.744557e+06
1,Furniture,4.110874e+06
2,Office Supplies,3.787070e+06


In [13]:
query = """
SELECT "Order ID", Category, Sales,
       ROW_NUMBER() OVER (PARTITION BY Category ORDER BY Sales DESC) AS rank_in_category
FROM orders;
"""
result = pd.read_sql(query, conn)
result.head(20)

,Order ID,Category,Sales,rank_in_category
0,ID-2012-34884,Furniture,5759.964,1
1,ES-2014-3785216,Furniture,5729.346,2
2,IN-2014-76016,Furniture,5667.870,3
3,IN-2014-56206,Furniture,5486.670,4
4,ES-2013-3939561,Furniture,5451.300,5
5,IN-2012-26274,Furniture,5451.300,6
6,IN-2012-86369,Furniture,5244.840,7
7,IN-2014-66615,Furniture,5049.000,8
8,ID-2012-28402,Furniture,4626.150,9
9,IT-2011-1978668,Furniture,4544.100,10


In [14]:
query = """
WITH ranked AS (
    SELECT Category, "Product Name", SUM(Sales) AS total_sales,
           RANK() OVER (PARTITION BY Category ORDER BY SUM(Sales) DESC) AS product_rank
    FROM orders
    GROUP BY Category, "Product Name"
)
SELECT * FROM ranked
WHERE product_rank <= 5
ORDER BY Category, product_rank;
"""
pd.read_sql(query, conn)

,Category,Product Name,total_sales,product_rank
0,Furniture,"Hon Executive Leather Armchair, Adjustable",58193.4841,1
1,Furniture,"Office Star Executive Leather Armchair, Adjust...",50661.6840,2
2,Furniture,"Harbour Creations Executive Leather Armchair, ...",50121.5160,3
3,Furniture,"SAFCO Executive Leather Armchair, Black",41923.5300,4
4,Furniture,"Novimex Executive Leather Armchair, Adjustable",40585.1336,5
5,Office Supplies,"Eldon File Cart, Single Width",34387.7287,1
6,Office Supplies,"Hoover Stove, White",32842.6043,2
7,Office Supplies,"Hoover Stove, Red",31663.7790,3
8,Office Supplies,"Rogers File Cart, Single Width",29466.3053,4
9,Office Supplies,"Smead Lockers, Industrial",28991.6640,5


In [15]:
query = """
WITH monthly_sales AS (
    SELECT strftime('%Y-%m', "Order Date") AS month, SUM(Sales) AS total_sales
    FROM orders
    GROUP BY month
)
SELECT month, total_sales,
       LAG(total_sales) OVER (ORDER BY month) AS prev_month_sales,
       total_sales - LAG(total_sales) OVER (ORDER BY month) AS change
FROM monthly_sales
ORDER BY month;
"""
pd.read_sql(query, conn)

,month,total_sales,prev_month_sales,change
0,2011-01,98898.48886,NaN,NaN
1,2011-02,91152.15698,98898.48886,-7746.33188
2,2011-03,145729.36736,91152.15698,54577.21038
3,2011-04,116915.76418,145729.36736,-28813.60318
4,2011-05,146747.83610,116915.76418,29832.07192
5,2011-06,215207.38022,146747.83610,68459.54412
6,2011-07,115510.41912,215207.38022,-99696.96110
7,2011-08,207581.49122,115510.41912,92071.07210
8,2011-09,290214.45534,207581.49122,82632.96412
9,2011-10,199071.26404,290214.45534,-91143.19130


In [16]:
query = """
CREATE VIEW IF NOT EXISTS category_summary AS
SELECT Category, SUM(Sales) AS total_sales, SUM(Profit) AS total_profit, COUNT(*) AS order_count
FROM orders
GROUP BY Category;
"""
conn.execute(query)
conn.commit()

# Now query the view like a table
pd.read_sql("SELECT * FROM category_summary;", conn)

,Category,total_sales,total_profit,order_count
0,Furniture,4.110874e+06,285204.72380,9876
1,Office Supplies,3.787070e+06,518473.83430,31273
2,Technology,4.744557e+06,663778.73318,10141


In [17]:
query = """
SELECT "Customer Name", SUM(Sales) AS total_spend, COUNT(*) AS num_orders
FROM orders
GROUP BY "Customer Name"
ORDER BY total_spend DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,Customer Name,total_spend,num_orders
0,Tom Ashbrook,40488.07080,80
1,Tamara Chand,37457.33300,88
2,Greg Tran,35550.95428,87
3,Christopher Conant,35187.07640,73
4,Sean Miller,35170.93296,50
5,Bart Watters,32310.44650,96
6,Natalie Fritzler,31781.25850,95
7,Fred Hopkins,30400.67452,82
8,Jane Waco,30288.45030,75
9,Hunter Lopez,30243.56658,53


In [18]:
query = """
SELECT Region, SUM(Profit) AS total_profit
FROM orders
GROUP BY Region
ORDER BY total_profit DESC;
"""
pd.read_sql(query, conn)

,Region,total_profit
0,Central,311403.98164
1,North,194597.95252
2,North Asia,165578.42100
3,South,140355.76618
4,Central Asia,132480.18700
5,Oceania,120089.11200
6,West,108418.44890
7,East,91522.78000
8,Africa,88871.63100
9,EMEA,43897.97100


In [19]:
query = """
SELECT "Ship Mode",
       COUNT(*) AS num_orders,
       AVG(julianday("Ship Date") - julianday("Order Date")) AS avg_days_to_ship
FROM orders
GROUP BY "Ship Mode"
ORDER BY num_orders DESC;
"""
pd.read_sql(query, conn)

,Ship Mode,num_orders,avg_days_to_ship
0,Standard Class,30775,4.998018
1,Second Class,10309,3.230187
2,First Class,7505,2.181746
3,Same Day,2701,0.037394


In [20]:
query = """
SELECT Discount, AVG(Profit) AS avg_profit, COUNT(*) AS num_orders
FROM orders
GROUP BY Discount
ORDER BY Discount;
"""
pd.read_sql(query, conn)

,Discount,avg_profit,num_orders
0,0.000,61.039514,29009
1,0.002,125.762649,461
2,0.070,140.990022,150
3,0.100,63.683426,4068
4,0.150,50.602409,541
5,0.170,38.317107,735
6,0.200,23.552594,4998
7,0.202,-14.518847,41
8,0.250,4.043371,198
9,0.270,-4.317213,388


In [21]:
query = """
SELECT "Customer Name", SUM(Profit) AS total_profit
FROM orders
GROUP BY "Customer Name"
ORDER BY total_profit DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

,Customer Name,total_profit
0,Tamara Chand,8672.89890
1,Raymond Buch,8453.04950
2,Sanjit Chand,8205.37990
3,Hunter Lopez,7816.56778
4,Bill Eplett,7410.00530


In [22]:
query = """
SELECT Segment, SUM(Sales) AS total_sales, SUM(Profit) AS total_profit, COUNT(*) AS num_orders
FROM orders
GROUP BY Segment
ORDER BY total_profit DESC;
"""
pd.read_sql(query, conn)

,Segment,total_sales,total_profit,num_orders
0,Consumer,6.507949e+06,749239.78206,26518
1,Corporate,3.824698e+06,441208.32866,15429
2,Home Office,2.309855e+06,277009.18056,9343


In [23]:
query = """
SELECT "Order Priority", AVG(Profit) AS avg_profit, COUNT(*) AS num_orders
FROM orders
GROUP BY "Order Priority"
ORDER BY avg_profit DESC;
"""
pd.read_sql(query, conn)

,Order Priority,avg_profit,num_orders
0,Critical,31.593124,3932
1,Medium,29.361729,29433
2,High,27.119122,15501
3,Low,24.197958,2424


In [24]:
query = """
WITH yearly_sales AS (
    SELECT strftime('%Y', "Order Date") AS year, SUM(Sales) AS total_sales
    FROM orders
    GROUP BY year
)
SELECT year, total_sales,
       LAG(total_sales) OVER (ORDER BY year) AS prev_year_sales,
       ROUND((total_sales - LAG(total_sales) OVER (ORDER BY year)) / LAG(total_sales) OVER (ORDER BY year) * 100, 2) AS growth_pct
FROM yearly_sales
ORDER BY year;
"""
pd.read_sql(query, conn)

,year,total_sales,prev_year_sales,growth_pct
0,2011,2.259451e+06,NaN,NaN
1,2012,2.677439e+06,2.259451e+06,18.50
2,2013,3.405746e+06,2.677439e+06,27.20
3,2014,4.299866e+06,3.405746e+06,26.25
